# 02 Transform Silver

This notebook builds the Silver layer from the Bronze layer.

Silver tables are cleaned, deduplicated and validated versions of the raw Bronze tables.

Current design:
- Bronze is append-oriented and preserves incoming files.
- Silver is rebuilt from the full Bronze layer using overwrite.
- Duplicate IDs are resolved by keeping the latest `ingestion_timestamp`.
- Orders are not dropped when references are missing. Instead, quality flags are added.


## 1. Setup

In [18]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, row_number, when
from pyspark.sql.window import Window

In [19]:
spark = (
    SparkSession.builder
    .appName("supply-chain-data-platform-silver")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [20]:
DATA_DIR = Path("../data")

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"

TABLES = ["orders", "customers", "products", "regions"]

for table in TABLES:
    (SILVER_DIR / table).mkdir(parents=True, exist_ok=True)

## 2. Read Bronze tables

In [21]:
bronze_orders = spark.read.parquet(str(BRONZE_DIR / "orders"))
bronze_customers = spark.read.parquet(str(BRONZE_DIR / "customers"))
bronze_products = spark.read.parquet(str(BRONZE_DIR / "products"))
bronze_regions = spark.read.parquet(str(BRONZE_DIR / "regions"))

In [22]:
bronze_tables = {
    "orders": bronze_orders,
    "customers": bronze_customers,
    "products": bronze_products,
    "regions": bronze_regions,
}

for table_name, df in bronze_tables.items():
    print(f"{table_name}: {df.count()} rows")

orders: 10000 rows
customers: 2000 rows
products: 500 rows
regions: 4 rows


## 3. Helper functions

Silver keeps the latest row per primary key. This supports the idea that later batches may contain newer versions of existing records.


In [23]:
def keep_latest_by_key(df, key_column: str):
    window = (
        Window
        .partitionBy(key_column)
        .orderBy(col("ingestion_timestamp").desc())
    )

    return (
        df
        .withColumn("_row_number", row_number().over(window))
        .filter(col("_row_number") == 1)
        .drop("_row_number")
    )


def drop_bronze_only_columns(df):
    columns_to_drop = [column for column in ["source_table"] if column in df.columns]
    return df.drop(*columns_to_drop)

## 4. Build Silver dimension tables

Customers, products and regions are deduplicated by primary key.

For now, Silver keeps:
- source columns
- `source_file`
- `ingestion_timestamp`

and drops:
- `source_table`


In [24]:
silver_customers = (
    bronze_customers
    .transform(lambda df: keep_latest_by_key(df, "customer_id"))
    .transform(drop_bronze_only_columns)
)

silver_products = (
    bronze_products
    .transform(lambda df: keep_latest_by_key(df, "product_id"))
    .transform(drop_bronze_only_columns)
)

silver_regions = (
    bronze_regions
    .transform(lambda df: keep_latest_by_key(df, "region_id"))
    .transform(drop_bronze_only_columns)
)

In [25]:
print("silver_customers:", silver_customers.count())
print("silver_products:", silver_products.count())
print("silver_regions:", silver_regions.count())

silver_customers: 2000
silver_products: 500
silver_regions: 4


## 5. Build Silver orders

Rules:
- Keep latest row per `order_id`.
- Drop `source_table`.
- Keep `source_file` and `ingestion_timestamp` for traceability.
- Add reference quality flags.
- Add value quality flags.
- Do not remove invalid orders yet.


In [26]:
silver_orders_base = (
    bronze_orders
    .transform(lambda df: keep_latest_by_key(df, "order_id"))
    .transform(drop_bronze_only_columns)
)

In [27]:
customer_refs = (
    silver_customers
    .select("customer_id")
    .dropDuplicates()
    .withColumn("has_valid_customer_reference", lit(True))
)

product_refs = (
    silver_products
    .select("product_id")
    .dropDuplicates()
    .withColumn("has_valid_product_reference", lit(True))
)

In [28]:
silver_orders = (
    silver_orders_base
    .join(customer_refs, on="customer_id", how="left")
    .join(product_refs, on="product_id", how="left")
    .fillna({
        "has_valid_customer_reference": False,
        "has_valid_product_reference": False,
    })
    .withColumn(
        "has_valid_order_values",
        col("order_id").isNotNull()
        & col("customer_id").isNotNull()
        & col("product_id").isNotNull()
        & col("order_date").isNotNull()
        & (col("quantity") > 0)
        & (col("total_amount") >= 0)
    )
    .withColumn(
        "is_valid_order",
        col("has_valid_customer_reference")
        & col("has_valid_product_reference")
        & col("has_valid_order_values")
    )
)

In [29]:
print("silver_orders:", silver_orders.count())

silver_orders.groupBy("has_valid_customer_reference").count().show()
silver_orders.groupBy("has_valid_product_reference").count().show()
silver_orders.groupBy("has_valid_order_values").count().show()
silver_orders.groupBy("is_valid_order").count().show()

silver_orders: 10000
+----------------------------+-----+
|has_valid_customer_reference|count|
+----------------------------+-----+
|                        true|10000|
+----------------------------+-----+

+---------------------------+-----+
|has_valid_product_reference|count|
+---------------------------+-----+
|                       true|10000|
+---------------------------+-----+

+----------------------+-----+
|has_valid_order_values|count|
+----------------------+-----+
|                  true|10000|
+----------------------+-----+

+--------------+-----+
|is_valid_order|count|
+--------------+-----+
|          true|10000|
+--------------+-----+



## 6. Write Silver tables

For this local version, Silver is rebuilt from the full Bronze layer and written with `overwrite`.

This is simple and deterministic for a small dataset. Later, this can be replaced by incremental merge/upsert logic using Delta Lake.


In [30]:
silver_orders.write.mode("overwrite").parquet(str(SILVER_DIR / "orders"))
silver_customers.write.mode("overwrite").parquet(str(SILVER_DIR / "customers"))
silver_products.write.mode("overwrite").parquet(str(SILVER_DIR / "products"))
silver_regions.write.mode("overwrite").parquet(str(SILVER_DIR / "regions"))

## 7. Validate Silver outputs

In [31]:
silver_outputs = {
    "orders": spark.read.parquet(str(SILVER_DIR / "orders")),
    "customers": spark.read.parquet(str(SILVER_DIR / "customers")),
    "products": spark.read.parquet(str(SILVER_DIR / "products")),
    "regions": spark.read.parquet(str(SILVER_DIR / "regions")),
}

for table_name, df in silver_outputs.items():
    print("=" * 80)
    print(table_name)
    print(f"Rows: {df.count()}")
    df.printSchema()

orders
Rows: 10000
root
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- has_valid_customer_reference: boolean (nullable = true)
 |-- has_valid_product_reference: boolean (nullable = true)
 |-- has_valid_order_values: boolean (nullable = true)
 |-- is_valid_order: boolean (nullable = true)

customers
Rows: 2000
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

products
Rows: 500
root
 |-- product_id: stri

In [32]:
silver_outputs["orders"].groupBy(
    "has_valid_customer_reference",
    "has_valid_product_reference",
    "has_valid_order_values",
    "is_valid_order",
).count().show(truncate=False)

+----------------------------+---------------------------+----------------------+--------------+-----+
|has_valid_customer_reference|has_valid_product_reference|has_valid_order_values|is_valid_order|count|
+----------------------------+---------------------------+----------------------+--------------+-----+
|true                        |true                       |true                  |true          |10000|
+----------------------------+---------------------------+----------------------+--------------+-----+



## 8. Notes

Current expected behavior after only the first batch is ingested into Bronze:
- Some orders may have invalid customer/product references because the missing customers/products may arrive in the second batch.
- This is not treated as a reason to drop rows in Silver.
- Instead, the invalid references are flagged.

Next step:
- Ingest the second batch into Bronze.
- Rerun this Silver notebook.
- Check whether the invalid reference counts decrease or disappear.
